In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)


In [29]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
dataset = pd.read_csv("estado_nuticional_sao_paulo.csv")

In [ ]:
dataset.head()

In [ ]:
dataset.tail()

In [ ]:
dataset.info()

Importamos uma base com informações da situação nutricional das pessoas da cidade de São Paulo/SP no período de Janeiro de 2022 à Dezembro de 2023 para treinar nosso modelo.

O objetivo é treinar o modelo para ele analisar dados como idade, altura, peso, etc... e definir em qual estado nutricional estão as pessoas, Magreza, Sobrepeso, Obesidade, etc...

Avaliando os primeiros e últimos registros da base de dados, podemos notar a preseça de algumas colunas irrelevantes para o treinamento do modelo, como códigos de sistemas e datas. 

Além disso, muitas colunas aparecem totalmente nulas e outras majoritariamente nulas, sendo que aplicar uma mediana não seria adequado porque são dados determiníticos, como 'DS_POVO_COMUNIDADE', que é a comunidade aonde a pessoa reside. Não tem uma técnica para "adivinhar" em qual comunidade a pessoa reside.

Abaixo temos a relação de colunas que vamos remover da base para trabalharmos somente com dados relevantes para o treinamento.

| Coluna | Motivo da Remoção |
| :--- | :--- |
| `ST_PARTICIPA_ANDI` | Dado irrelevante (participação no programa ANDI). |
| `CO_POVO_COMUNIDADE` | Majoritariamente nulo. |
| `DS_IMC_PRE_GESTACIONAL` | Todos os registros são nulos. |
| `CO_ESCOLARIDADE` | Majoritariamente nulo. |
| `DS_ESCOLARIDADE` | Maioria dos dados consta como "SEM INFORMAÇÃO". |
| `CO_ACOMPANHAMENTO` | Código técnico/sistema irrelevante para o modelo. |
| `CO_PESSOA_SISVAN` | Código de identificação irrelevante. |
| `CO_MUNICIPIO_IBGE` | Valor constante (Todos são SÃO PAULO). |
| `SG_UF` | Valor constante (Todos são SÃO PAULO). |
| `NO_MUNICIPIO` | Valor constante (Todos são SÃO PAULO). |
| `CO_CNES` | Código de estabelecimento irrelevante. |
| `DS_POVO_COMUNIDADE` | Maioria dos dados consta como "NÃO INFORMADO". |
| `NU_COMPETENCIA` | Referência temporal (mês/ano) irrelevante para o estado nutricional. |
| `CO_SISTEMA_ORIGEM_ACOMP` | Código de sistema irrelevante. |
| `SISTEMA_ORIGEM_ACOMP` | Informação de origem do dado irrelevante. |
| `DT_ACOMPANHAMENTO` | Data do registro, irrelevante para a predição atual. |
| `PESO X IDADE` | Específico apenas para faixa etária de 0 a 10 anos. |
| `CRI. ALTURA X IDADE` | Específico apenas para faixa etária de 0 a 10 anos. |
| `ADO. ALTURA X IDADE` | Específico apenas para faixa etária de 10 a menos de 20 anos. |
| `PESO X ALTURA` | Informação redundante (contida na coluna CRI. IMC X IDADE). |  

OBS.: Qualquer dúvida sobre os dados, vide o arquivo 'Dicionário_de_Dados_Estado_Nutricional.pdf' que está inclúso neste projeto.

In [ ]:
gestantes = dataset["CO_ESTADO_NUTRI_IMC_SEMGEST"].notna()
total_gestantes = gestantes.sum()
total_dataset = len(dataset)
total_nao_gestantes = total_dataset - total_gestantes

labels = ['Não Gestantes (População Geral)', 'Gestantes']
sizes = [total_nao_gestantes, total_gestantes]
colors = ['#4c72b0', '#ff9999'] # Azul padrão e um Vermelho/Rosa para destaque
explode = (0, 0.2)  # "Puxa" a fatia das gestantes para fora para dar ênfase

plt.figure(figsize=(10, 7))
plt.pie(sizes, 
        explode=explode, 
        labels=labels, 
        colors=colors, 
        autopct='%1.1f%%', # Mostra a porcentagem com uma casa decimal
        shadow=True, 
        startangle=140,
        textprops={'fontsize': 12, 'fontweight': 'bold'})

plt.title(f'Representatividade de Gestantes na Base Total\n(N = {total_dataset:,})', fontsize=15, pad=20)
plt.axis('equal')

plt.tight_layout()
plt.show()


Talvez possamos eliminar gestantes. O Dicionarios de dados diz q esta atualizado até 2021. Mas estamos atualizando 2022 em diante. O Estado de nutricao da gestante também é impactado pela semana de gestacao e não temos esses dados.

In [30]:
clean_dataset = dataset.drop(columns=[
                                "ST_PARTICIPA_ANDI", # Irrelevante: Se o municipio participa do programa ANDI
                                "CO_POVO_COMUNIDADE", # A maioria é nulo
                                "DS_IMC_PRE_GESTACIONAL", # Todos nulos
                                "CO_ESCOLARIDADE", # A maioria é nulo
                                "DS_ESCOLARIDADE", # A maioria é SEM INFORMAÇÃO
                                "CO_ACOMPANHAMENTO", # Código irrelevante
                                "CO_PESSOA_SISVAN", # Código irrelevante
                                "CO_MUNICIPIO_IBGE", # Todos são SÃO PAULO
                                "SG_UF", # Todos são SÃO PAULO
                                "NO_MUNICIPIO", # Todos são SÃO PAULO
                                "CO_CNES", # Código irrelevante
                                "DS_POVO_COMUNIDADE", # A maioria é NÃO INFORMADO
                                "NU_COMPETENCIA", # Ano/Mês do acompanhamento, irrelevante neste caso
                                "CO_SISTEMA_ORIGEM_ACOMP", # Código irrelevante
                                "SISTEMA_ORIGEM_ACOMP", # Informação irrelevante
                                "DT_ACOMPANHAMENTO", # Data, irrelevante neste caso
                                "PESO X IDADE", # Somente para as idades de 0 a 10 anos
                                "CRI. ALTURA X IDADE", # Somente de 0 a 10 anos
                                "ADO. ALTURA X IDADE", # Somente de 10 a menos de 20 anos
                                "PESO X ALTURA", # Tem na coluna CRI. IMC X IDADE
                                ])

In [ ]:
clean_dataset.info()

Corrige tipos das colunas que deveriam ser numeros

In [31]:
for col in ['NU_PESO', 'NU_ALTURA', 'DS_IMC']:
    clean_dataset[col] = clean_dataset[col].astype(str).str.replace(',', '.').astype(float)

Muda valores de sexo para inteiros

In [32]:
clean_dataset['SG_SEXO_NUM'] = clean_dataset['SG_SEXO'].map({'M': 0, 'F': 1})

In [ ]:
clean_dataset.info()

In [ ]:
clean_dataset.head()

Com base limpa, podemos ver que as colunas restantes conetem bem menos informações nulas e nenhuma irrelevante para o treinamento.

Abaixo temos a relação de colunas variáveis, que são os dados a serem analisados para chegar na conclusão final que é o estado nutricional da pessoa.
Note que temos todos os dados preenchidos sem nulos em nenhum registro.

| # | Column | Non-Null Count | Dtype |
|---| :--- | :--- | :--- |
| 0 | NU_IDADE_ANO | 1.582.936 | int64 |
| 1 | NU_FASE_VIDA | 1.582.936 | float64 |
| 2 | DS_FASE_VIDA | 1.582.936 | str |
| 3 | SG_SEXO | 1.582.936 | str |
| 4 | CO_RACA_COR | 1.582.936 | int64 |
| 5 | DS_RACA_COR | 1.582.936 | str |
| 6 | NU_PESO | 1.582.936 | float64 |
| 7 | NU_ALTURA | 1.582.936 | float64 |
| 8 | DS_IMC | 1.582.936 | float64 |

 Os dados como idade, fase de vida, sexo, raça/cor, peso, altura e IMC serão analisados para chegar na conclusão final, nosso alvo (target).

 Anaisando os dados atuais, após a limpeza, notamos que nosso target está distribuído em 5 colunas diferentes que se diferenciam pela faixa etária dos indivíduos.

| # | Column | Non-Null Count | Dtype |
|---| :--- | :--- | :--- |
| 9 | CRI. IMC X IDADE | 347.088 | str |
| 10 | ADO. IMC X IDADE | 190.113 | str |
| 11 | CO_ESTADO_NUTRI_ADULTO | 725.655 | str |
| 12 | CO_ESTADO_NUTRI_IDOSO | 309.080 | str |
| 13 | CO_ESTADO_NUTRI_IMC_SEMGEST | 38.933 | str |

A coluna 'CRI. IMC X IDADE' traz informações de pessoas entre 0 a 10 anos, aonde os valores são: 'Magreza acentuada’, ‘Magreza’, ‘Eutrofia', 'Riscode sobrepeso', 'Sobrepeso', 'Obesidade'. Nas demais pessoas fora desta faixa de idade, essa coluna tera a informação nula.

Ja a coluna 'ADO. IMC X IDADE' traz informações de pessoas entre 10 e 20 anos, aonde os valores são: Magreza acentuada', 'Magreza', 'Eutrofia', 'Risco de sobrepeso','Sobrepeso', 'Obesidade'. As pessoas fora desta faixa de idade terão essa coluna nula.

A coluna 'CO_ESTADO_NUTRI_ADULTO' traz informações de pessoas entre 20 e 60 anos, aonde os valores são: 'Baixo peso', 'Adequado ou Eutrófico', 'Sobrepeso', 'Obesidade Grau I','Obesidade Grau II', 'Obesidade Grau III'. Note aqui o resultado é um pouco diferente, a obesidade se divide em 3 graus. Isso porque só é possível medir com essa precisão em pessoas nesta faixa etária.

A colina 'CO_ESTADO_NUTRI_IDOSO' traz informações de pessoas com 60 anos ou mais, aonde os valores são: 'Baixo peso', 'Adequado ou Eutrófico','Sobrepeso'

Agora vamos analisar se todos os 1.582.936 registros possuem alguma das informações alvo (target) sem a informação de gestantes (CO_ESTADO_NUTRI_IMC_SEMGEST).

In [ ]:
filtro = (
    clean_dataset['CRI. IMC X IDADE'].isna() & 
    clean_dataset['ADO. IMC X IDADE'].isna() & 
    clean_dataset['CO_ESTADO_NUTRI_ADULTO'].isna() & 
    clean_dataset['CO_ESTADO_NUTRI_IDOSO'].isna()
)

print(f"Total sem NENHUM target: {len(clean_dataset[filtro])}")

Ficamos com 11000 registros sem o target.

Agora vamos incluir a informação de gestantes para ver se conseguimos mais alguns registros com o target.

In [ ]:
filtro = (
    clean_dataset['CRI. IMC X IDADE'].isna() & 
    clean_dataset['ADO. IMC X IDADE'].isna() & 
    clean_dataset['CO_ESTADO_NUTRI_ADULTO'].isna() & 
    clean_dataset['CO_ESTADO_NUTRI_IDOSO'].isna() &
    clean_dataset['CO_ESTADO_NUTRI_IMC_SEMGEST'].isna()
)

print(f"Total sem NENHUM target: {len(clean_dataset[filtro])}")

Com esta coluna conseguimos diminuir os registros sem target para 10650 (350 que não tinham nenum dos 4 targets foram classificados com a informação de 'CO_ESTADO_NUTRI_IMC_SEMGEST').

Vamos utilizar esta coluna para classificar os indivíduos que não se enquadram nas ontras colunas.

### Classificação do Estado Nutricional por Grupo (Targets)

*   **CRI. IMC X IDADE** (0 a 10 anos)
    *   **Valores:** `Magreza acentuada`, `Magreza`, `Eutrofia`, `Risco de sobrepeso`, `Sobrepeso`, `Obesidade`.
*   **ADO. IMC X IDADE** (10 a menos de 20 anos)
    *   **Valores:** `Magreza acentuada`, `Magreza`, `Eutrofia`, `Risco de sobrepeso`, `Sobrepeso`, `Obesidade`.
*   **CO_ESTADO_NUTRI_ADULTO** (20 a 60 anos)
    *   **Valores:** `Baixo peso`, `Adequado ou Eutrófico`, `Sobrepeso`, `Obesidade Grau I`, `Obesidade Grau II`, `Obesidade Grau III`.
*   **CO_ESTADO_NUTRI_IDOSO** (60 anos ou mais)
    *   **Valores:** `Baixo peso`, `Adequado ou Eutrófico`, `Sobrepeso`.
*   **CO_ESTADO_NUTRI_IMC_SEMGEST** (Gestantes de 10 a 60 anos)
    *   **Valores:** `Baixo peso`, `Adequado ou Eutrófico`, `Sobrepeso`, `Obesidade`.
    *   *Nota: Aplicar caso não conste em nenhuma das categorias acima.*

Com esta análise, podemos juntar os targets em uma única coluna unifique os estados de nutrição

| # | Descrição | Grupo SISVAN |
| :--- | :--- | :--- |
| 0 | Magreza Acentuada | 'Magreza acentuada' |
| 1 | Baixo Peso | 'Magreza', 'Baixo peso' |
| 2 | Eutrofia (Normal) | 'Eutrofia', 'Adequado ou Eutrófico' |
| 3 | Risco/Sobrepeso | 'Risco de sobrepeso', 'Sobrepeso' |
| 4 | Obesidade | 'Obesidade', 'Obesidade Grau I' |
| 5 | Obesidade Grave | 'Obesidade Grau II', 'Obesidade Grau III' |



In [33]:
nome_novas_classes = {
    0: 'Magreza Acentuada.', 
    1: 'Baixo Peso', 
    2: 'Eutrofia', 
    3: 'Sobrepeso', 
    4: 'Obesidade', 
    5: 'Obesidade II',
    6: 'Obesidade III'
}

mapeamento_classes = {
    'Magreza acentuada': 0,
    
    'Magreza': 1, 
    'Baixo peso': 1,
    
    'Eutrofia': 2, 
    'Adequado ou eutrófico': 2,
    'Adequado ou Eutrófico': 2,
    'Peso Adequado ou Eutrofico': 2,
    
    'Risco de sobrepeso': 3, 
    'Sobrepeso': 3,
    
    'Obesidade': 4, 
    'Obesidade Grau I': 4,
    
    'Obesidade Grau II': 5, 
    'Obesidade Grau III': 6
}

colunas_diagnostico = [
    'CO_ESTADO_NUTRI_IMC_SEMGEST', # Gestante (tem q ser primeiro, pq mesmo sendo adulto se for gestante é tem q usar valor dessa coluna)
    'CO_ESTADO_NUTRI_ADULTO',      # Adulto
    'CO_ESTADO_NUTRI_IDOSO',       # Idoso
    'ADO. IMC X IDADE',            # Adolescente
    'CRI. IMC X IDADE'             # Criança
]

def definir_alvo(row):
    for col in colunas_diagnostico:
        valor = row[col]
        if pd.notnull(valor) and str(valor).strip() != '':
            return mapeamento_classes.get(str(valor).strip())
    
    return np.nan

clean_dataset['ESTADO_NUTRI'] = dataset.apply(definir_alvo, axis=1)

In [ ]:
clean_dataset.info()


O grafico a seguir mostra os outliers do IMC. IMC acima de 120 é improvavel. Tem um espaço muito grande entre ~60 e 100. Isso é um erro de digitação? 

In [ ]:
plt.figure(figsize=(12, 6))

sns.set_theme(style="whitegrid")
sns.histplot(data=clean_dataset, 
             x='DS_IMC',
             kde=True,        
             bins=50,         
             color='skyblue',
             edgecolor='black')

plt.axvline(clean_dataset['DS_IMC'].mean(), color='red', linestyle='--', label=f'Média: {clean_dataset["DS_IMC"].mean():.2f}')
plt.axvline(clean_dataset['DS_IMC'].median(), color='green', linestyle='-', label=f'Mediana: {clean_dataset["DS_IMC"].median():.2f}')

plt.title('Distribuição da Frequência do IMC', fontsize=16)
plt.xlabel('Índice de Massa Corporal (IMC)', fontsize=12)
plt.ylabel('Quantidade de Registros', fontsize=12)
plt.legend()

plt.show()

Com a nova coluna 'ESTADO_NUTRI' consolidada, podemos ver que 1572286 dos 1582936 registros foram classificados com sucesso.

Com a nova coluna target estabelecida, podemos remover as demais que foram usadas para gerá-la.

In [ ]:
clean_target_dataset = clean_dataset.drop(columns=[
                                                    "CRI. IMC X IDADE",
                                                    "ADO. IMC X IDADE",
                                                    "CO_ESTADO_NUTRI_ADULTO",
                                                    "CO_ESTADO_NUTRI_IDOSO",
                                                    "CO_ESTADO_NUTRI_IMC_SEMGEST"
                                                    ])

clean_target_dataset.info()

In [ ]:
clean_target_dataset.head()

Com a base quase limpa e com um único target estabelecido, vamos ver quantos e quais registros não conseguiram nenhuma classificação.

In [ ]:
print(f"Quantidade de registros SEM classificação(ESTADO_NUTRI): {len(clean_target_dataset[clean_target_dataset["ESTADO_NUTRI"].isna()])}")

In [ ]:
labels = [f"{i}. {desc}" for i, desc in nome_novas_classes.items()]

sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))

ax = sns.countplot(
    x=clean_dataset["ESTADO_NUTRI"], 
    hue=clean_dataset["ESTADO_NUTRI"], 

    palette="viridis", 
    legend=False
)
ax.set_xticklabels(labels, rotation=15)

total = len(clean_dataset)
for p in ax.patches:
    percentage = '{:.1f}%'.format(100 * p.get_height()/total)
    x = p.get_x() + p.get_width() / 2 - 0.15
    y = p.get_height() + (total * 0.01)
    ax.annotate(percentage, (x, y), fontsize=11, fontweight='bold')

plt.title('Distribuição do Estado Nutricional (Target)', fontsize=15)
plt.xlabel('Classes (0 a 5)', fontsize=12)
plt.ylabel('Quantidade de Pessoas', fontsize=12)
plt.tight_layout()
plt.show()

O grafico anterior mostra a distruição dos dados. Ele mostra que há poucos exemplos de magreza acentuada isso pode afetar o modelo quando for classificar esse tipo, A gnete pode unificar baixo peso e magreza acentuada. O mesmo para Obesidade II e III.

Mas podemos tem problemas, porque os dois extremos é exatamente onde requer mais cuidados médicos.


In [ ]:
clean_target_dataset[clean_target_dataset["ESTADO_NUTRI"].isna()][:20]

Como a base possui mais de 1.5 milhões de registro, remover esta pequena quantidade de 10650 sem classificação não vai prejudicar o trenamento do modelo.

Vamos fazer isso e analisar os dados novamente.

In [ ]:
clean_target_dataset.dropna(subset=['ESTADO_NUTRI'], inplace=True)

clean_target_dataset.info()

O grafico a seguir mostra que cor/raça tem baixa relevância (as barras tem tamanhos e coloracao semelhantes) no estado de nutricao. Portanto 'DS_RACA_COR' pode ser removida

In [ ]:
prop_raca = pd.crosstab(clean_dataset["DS_RACA_COR"], clean_dataset["ESTADO_NUTRI"], normalize='index') * 100

prop_raca.plot(kind='bar', stacked=True, figsize=(12, 7), colormap='viridis')
plt.title('Composição do Estado Nutricional por Raça/Cor (%)', fontsize=15)
plt.legend(title='Estado Nutri', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.ylabel('Porcentagem (%)')
plt.xlabel('Raça/Cor')
plt.xticks(rotation=45)
plt.show()

Os gráficos a seguir mostram a relação entre o IMC e a Idade. No primeiro, é possível identificar outliers, como IMCs próximos a 200, o que é 'impossível' e talvez um erro de digital. Acho q é melhor  manter apenas os registros onde o IMC está entre 10 e 100.

Também exibimos as 'Fases da Vida'. Decidimos manter essa coluna para o modelo não precisar 'adivinhar' as mudanças biológicas sozinho; com ela, garantimos que a divisão dos grupos (criança, adulto, idoso) funcione perfeitamente.

A Fase da Vida ajuda o modelo a entender o desbalanceamento: crianças e idosos têm pouquíssimos casos de IMC muito baixo ou muito alto.

In [ ]:
plt.figure(figsize=(12, 6))
sns.scatterplot(data=clean_dataset.sample(100000), x='NU_IDADE_ANO', y='DS_IMC', hue='ESTADO_NUTRI', palette='viridis', alpha=0.5)
plt.title('Relação Idade x IMC colorida pelo Target')
plt.show()

In [ ]:
prop_fase_vida = pd.crosstab(clean_dataset["NU_FASE_VIDA"], clean_dataset["ESTADO_NUTRI"], normalize='index') * 100

prop_fase_vida.plot(kind='bar', stacked=True, figsize=(12, 7), colormap='viridis')
plt.title('Composição do Estado Nutricional por Fase vida (%)', fontsize=15)
plt.legend(title='Estado Nutri', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.ylabel('Porcentagem (%)')
plt.xlabel('Fase Vida')
plt.xticks(rotation=45)
plt.show()

In [ ]:
clean_target_dataset[:20]

Agora sim, com a base limpa, aonde temos todos os dados com valores e alvos informados, vamos trabalhar na normalização dos dados, tranformando o que texto em números.

Para isso, primeiro vamos remover os códigos existentes (NU_FASE_VIDA e CO_RACA_COR) e deixar somente os textos (DS_FASE_VIDA e DS_RACA_COR) para aplicarmos a alteração via recursos da Sk Learn.

In [ ]:
clean_target_dataset.drop(columns=[
                                    "NU_FASE_VIDA",
                                    "CO_RACA_COR"
                                    ], inplace=True)

clean_target_dataset[:20]

### Analisando as colunas referentes à faixa etária de 0 a 10 anos

As colunas alvo desta faixa etária (conforme descrito na documentação) apresentam quantidade de dados diferentes. Vamos avaliar a distribuição da idade entre as colunas para entendimento dos dados.

19  PESO X IDADE             347146 non-null   str    
20  PESO X ALTURA            236479 non-null   str    
21  CRI. ALTURA X IDADE      347154 non-null   str    
22  CRI. IMC X IDADE         347088 non-null   str  

In [ ]:
df_peso_idade = dataset.loc[dataset['PESO X IDADE'].notna(), ['NU_IDADE_ANO']]
df_peso_altura = dataset.loc[dataset['PESO X ALTURA'].notna(), ['NU_IDADE_ANO']]
df_cri_altura_idade = dataset.loc[dataset['CRI. ALTURA X IDADE'].notna(), ['NU_IDADE_ANO']]
df_cri_imc_idade = dataset.loc[dataset['CRI. IMC X IDADE'].notna(), ['NU_IDADE_ANO']]

In [ ]:
df_peso_altura.value_counts()

In [ ]:
df_cri_altura_idade.value_counts()

In [ ]:
df_cri_imc_idade.value_counts()

Avaliando-se a faixa etária coberta por cada coluna, nota-se que a coluna "PESO X ALTURA", que de acordo com o dicionário de dados deveria cobrir a mesma faixa etária das demais, tem menos dados. Vamos analisar se estes dados aparecem em conjunto com as outras colunas nos mesmos registros ou se estão em amostras separadas.

In [ ]:
df_all_filled = dataset.loc[dataset['PESO X ALTURA'].notna() & dataset['PESO X IDADE'].notna() & dataset['CRI. ALTURA X IDADE'].notna() & dataset['CRI. IMC X IDADE'].notna(), ['NU_IDADE_ANO'] ]
df_1_null = dataset.loc[dataset['PESO X ALTURA'].notna() & (dataset['PESO X IDADE'].isna() | dataset['CRI. ALTURA X IDADE'].isna() | dataset['CRI. IMC X IDADE'].isna()), ['NU_IDADE_ANO'] ]

In [ ]:
print(len(df_all_filled))
print(len(df_1_null))

Baseado nos números de dados preenchidos, a coluna PESO x ALTURA apresenta um dado específico para crianças de 0 a 6 anos e está preenchida em conjunto com as demais colunas da faixa etária de 0 a 8 anos.

### Adicionando o cálculo do % Gordura a partir da fórmula de Deuremberg, que leva em conta idade e sexo.

Para Adultos: 1.51 x IMC - 0.70 x Idade - 3.6 x sexo + 1.4 (R² 0.38 e SEE 4%, segundo o documento original )
Para crianças até 15 anos: 1.20 x IMC + 0.23 x Idade - 10.8 x sexo - 5.4 (R² 0.79 e SEE 4.1%, segundo o documento original )
 
Onde: Masculino = 1 e Feminino = 0

#### Classificação do Percentual de Gordura Corporal

**Homens**
| Classificação        | 20–29 anos | 30–39 anos | 40–49 anos | 50–59 anos | 60+ anos |
|----------------------|------------|------------|------------|------------|----------|
| Excelente            | < 11%      | < 12%      | < 14%      | < 15%      | < 16%    |
| Bom                  | 11–13%     | 12–15%     | 14–17%     | 15–18%     | 16–19%   |
| Acima da média       | 14–20%     | 16–21%     | 18–23%     | 19–24%     | 20–25%   |
| Abaixo da média      | 21–23%     | 22–24%     | 24–26%     | 25–27%     | 26–28%   |
| Ruim                 | > 23%      | > 24%      | > 26%      | > 27%      | > 28%    |

**Mulheres**
| Classificação        | 20–29 anos | 30–39 anos | 40–49 anos | 50–59 anos | 60+ anos |
|----------------------|------------|------------|------------|------------|----------|
| Excelente            | < 16%      | < 17%      | < 18%      | < 19%      | < 20%    |
| Bom                  | 16–19%     | 17–20%     | 18–21%     | 19–22%     | 20–23%   |
| Acima da média       | 20–28%     | 21–29%     | 22–30%     | 23–31%     | 24–32%   |
| Abaixo da média      | 29–31%     | 30–32%     | 31–33%     | 32–34%     | 33–35%   |
| Ruim                 | > 31%      | > 32%      | > 33%      | > 34%      | > 35%    |




Fontes:

**DEURENBERG, P.; WESTSTRATE, J. A.; SEIDELL, J. C.** Body mass index as a measure of
body fatness: age- and sex-specific prediction formulas. *British Journal of Nutrition*,
Cambridge, v. 65, n. 2, p. 105–114, mar. 1991.
DOI: [10.1079/BJN19910073](https://doi.org/10.1079/BJN19910073).
Disponível em: [https://pubmed.ncbi.nlm.nih.gov/2043597/](https://pubmed.ncbi.nlm.nih.gov/2043597/).
Acesso em: 01 abr. 2026.

American College of Sports Medicine. ACSM's Health-Related Physical Fitness Assessment Manual. 2nd ed. Philadelphia: Lippincott Williams & Wilkins, 2008. p. 59.

> OBS: a fórmla de Deuremberg foi desenvolvida principalmente em populações caucasianas européias, então deve-se considerar alguma margem de erro em populações como a do Brasil, que tem maior diversidade étnica.

In [ ]:
clean_target_dataset.head()

In [ ]:
clean_target_dataset['PERC_GORDURA'] = np.where(
    clean_target_dataset['NU_IDADE_ANO'] <=15,
    1.51 * clean_target_dataset['DS_IMC'] - 0.70 * clean_target_dataset['NU_IDADE_ANO'] - 3.6 * (1 - clean_target_dataset['SG_SEXO_NUM'] + 1.4),
    1.20 * clean_target_dataset['DS_IMC'] - 0.23 * clean_target_dataset['NU_IDADE_ANO'] - 10.8 * (1 - clean_target_dataset['SG_SEXO_NUM'] - 5.4))

clean_target_dataset['PERC_GORDURA'] = clean_target_dataset['PERC_GORDURA'].round(2)

clean_target_dataset.head()

### Convertendo as variáveis categoricas

In [ ]:
dist_fases_vida = clean_target_dataset['DS_FASE_VIDA'].value_counts() / len(clean_target_dataset) * 100
dist_fases_vida

Analisando a distribuição da variável "DS_FASES_VIDA", podemos notar que há uma concentração dos registros na fase adulta, com 46% da base. Deve-se levar em conta esta característia no momento da separação dos dados de treino e testes.

DS_FASE_VIDA
ADULTO                    46.171562
IDOSO                     19.661563
ADOLESCENTE               12.091502
ENTRE 6 MESES A 2 ANOS     6.401698
ENTRE 2 ANOS A 5 ANOS      6.145510
ENTRE 7 ANOS A 10 ANOS     3.852861
ENTRE 5 ANOS A 7 ANOS      3.389841
MENOR DE 6 MESES           2.285462
Name: count, dtype: float64

In [ ]:
dist_raca_cor = clean_target_dataset['DS_RACA_COR'].value_counts() / len(clean_target_dataset) * 100
dist_raca_cor

Aqui a análise de Raça/Cor indica que aprox. 45% da base não tem a informação preenchida, portanto pode causar alguma distorção dos resultados caso usemos esta variável no treinamento.

DS_RACA_COR
SEM INFORMACAO    44.764502
BRANCA            25.859036
PARDA             16.508002
AMARELA            9.038750
PRETA              3.714210
INDIGENA           0.115501
Name: count, dtype: float64

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
hierarquia=[['MENOR DE 6 MESES','ENTRE 6 MESES A 2 ANOS','ENTRE 2 ANOS A 5 ANOS','ENTRE 5 ANOS A 7 ANOS','ENTRE 7 ANOS A 10 ANOS','ADOLESCENTE','ADULTO','IDOSO']] # passar como parâmetro para o encoder para respeitar a order de hierarquia das fases

ordinalEncoder = OrdinalEncoder(categories=hierarquia)

clean_target_dataset['CD_FASE_VIDA'] = ordinalEncoder.fit_transform(clean_target_dataset[['DS_FASE_VIDA']])

clean_target_dataset[['DS_FASE_VIDA','CD_FASE_VIDA']].head(10)

Amostra da classificação pelo OrdinalEncoder

| # | DS_FASE_VIDA | CD_FASE_VIDA |
| :-- | :-- | :-- |
| 0 |	ADULTO	| 6.0 |
| 1 |	IDOSO	| 7.0 |
| 2 |	ADULTO	| 6.0 |
| 3 |	ENTRE 6 MESES A 2 ANOS	| 1.0 |
| 4 |	ADULTO	| 6.0 |
| 5 |	ADULTO	| 6.0 |
| 6 |	ADULTO	| 6.0 |
| 7 |	ADULTO	| 6.0 |
| 8 |	ADULTO	| 6.0 |
| 9 |	ADULTO	| 6.0 |

In [ ]:
mapeamento_fases_vida = {
    categoria : indice for indice, categoria in enumerate(ordinalEncoder.categories_[0])
}

mapeamento_fases_vida